<a href="https://colab.research.google.com/github/corrielynnyuill-debug/Assignment13-CLY/blob/main/Assignment13_CorrieLynnYuill.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import tensorflow as tf
import numpy as np
import requests
import re

# Download Alice in Wonderland from Project Gutenberg
url = 'https://www.gutenberg.org/files/11/11-0.txt'
response = requests.get(url)
text = response.text

print('Raw length:', len(text))
print(text[:500])

# Clean Text
# Remove header and footer
start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
end_marker ="*** END OF THE PROJECT GUTENBERG EBOOK"

start = text.find(start_marker)
end = text.find(end_marker)

if start == -1 and end == -1:
  raise ValueError("Start and end markers not found in the text.")

text = text[start:end]

text = '\n'.join(text.split('\n')[1:])

# Normalize whitespace
text = re.sub(r'\s+', ' ', text).strip()

print('Clean length:', len(text))
print(text[:500])

# Character Tokenization
# Create character vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
print('Vocab size:', vocab_size)

# Mapping char to int
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}

# Encode entire text
encoded_text = [char_to_idx[c] for c in text]

# Create Training Sequence
seq_length = 100
step = 1

X = []
y = []

for i in range(0, len(encoded_text)-seq_length, step):
  X.append(encoded_text[i:i+seq_length])
  y.append(encoded_text[i+seq_length])

X = np.array(X)
y = np.array(y)

# Build a simple LSTM Model
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, 64, input_length=seq_length),
    tf.keras.layers.LSTM(512),
    tf.keras.layers.Dense(vocab_size, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam')
model.summary()

# Train the model
history = model.fit(X, y, epochs=30, batch_size=128)

print('-'*80)

Raw length: 144696
*** START OF THE PROJECT GUTENBERG EBOOK 11 ***

[Illustration]




Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race and a Long Tale
 CHAPTER IV.    The Rabbit Sends in a Little Bill
 CHAPTER V.     Advice from a Caterpillar
 CHAPTER VI.    Pig and Pepper
 CHAPTER VII.   A Mad Tea-Party
 CHAPTER VIII.  The Queen’s Croquet-Ground
 CHAPTER IX.    The
Clean length: 143127
[Illustration] Alice’s Adventures in Wonderland by Lewis Carroll THE MILLENNIUM FULCRUM EDITION 3.0 Contents CHAPTER I. Down the Rabbit-Hole CHAPTER II. The Pool of Tears CHAPTER III. A Caucus-Race and a Long Tale CHAPTER IV. The Rabbit Sends in a Little Bill CHAPTER V. Advice from a Caterpillar CHAPTER VI. Pig and Pepper CHAPTER VII. A Mad Tea-Party CHAPTER VIII. The Queen’s Croquet-Ground CHAPTER IX. The Mock Turtle’s Story CHAPTER X. The Lobster Quadri

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
1118/1118 ━━━━━━━━━━━━━━━━━━━━ 963s 857ms/step - loss: 2.3195
Epoch 2/10
1118/1118 ━━━━━━━━━━━━━━━━━━━━ 961s 860ms/step - loss: 1.8287
Epoch 3/10
1118/1118 ━━━━━━━━━━━━━━━━━━━━ 952s 852ms/step - loss: 1.6200
Epoch 4/10
1118/1118 ━━━━━━━━━━━━━━━━━━━━ 989s 858ms/step - loss: 1.4803
Epoch 5/10
1118/1118 ━━━━━━━━━━━━━━━━━━━━ 976s 853ms/step - loss: 1.3817
Epoch 6/10
1118/1118 ━━━━━━━━━━━━━━━━━━━━ 958s 857ms/step - loss: 1.3069
Epoch 7/10
1118/1118 ━━━━━━━━━━━━━━━━━━━━ 953s 853ms/step - loss: 1.2482
Epoch 8/10
1118/1118 ━━━━━━━━━━━━━━━━━━━━ 986s 856ms/step - loss: 1.1979
Epoch 9/10
1118/1118 ━━━━━━━━━━━━━━━━━━━━ 963s 861ms/step - loss: 1.1540
Epoch 10/10
1118/1118 ━━━━━━━━━━━━━━━━━━━━ 966s 846ms/step - loss: 1.1151
--------------------------------------------------------------------------------


In [14]:
# Text generation function
def sample(preds, temperature=0.7):
  preds = np.array(preds).astype('float64')
  preds = np.log(preds + 1e-8) / temperature
  exp_preds = np.exp(preds)
  preds = exp_preds / np.sum(exp_preds)
  return np.random.choice(len(preds), p=preds)

def generate_text(seed, length=400):
    generated = seed
    sequence = np.array([char_to_idx[c] for c in seed])

    for _ in range(length):
        # Prepare integer input (no one-hot)
        x = sequence[-seq_length:]
        if len(x) < seq_length:
            x = np.pad(x, (seq_length - len(x), 0))
        x = np.expand_dims(x, axis=0)

        # Predict next character
        preds = model.predict(x, verbose=0)[0]
        next_idx = np.random.choice(range(vocab_size), p=preds)
        next_idx = sample(preds, temperature=0.7)
        next_char = idx_to_char[next_idx]

        generated += next_char
        sequence = np.append(sequence, next_idx)

    return generated

print('-'*80)

--------------------------------------------------------------------------------


In [15]:
seed = 'Alice was beginning to get very tired '
print(generate_text(seed))

Alice was beginning to get very tired to herself by trying to be to the best and difficul) of her simple about to Alice’s voice again: again the Mouse, dearing her good liked left panting in watching itself down, planit vilence shouting, and hengurely great dived, when side of pleasled into a bown of but, certainly to Alice’s took gonded, the only one knew for smanking took plankingilar, but she first fartenesly gave to her of ly live
